# FastAPI Assignment - 5

## Objective
To create REST APIs using FastAPI including:
- Query Parameters
- Path Parameters
- Request Body
- Basic Logic Implementation

## Tools Used
- Python
- FastAPI
- Uvicorn

## How to Run
1. Install dependencies:
   pip install fastapi uvicorn

2. Run server:
   uvicorn main:app --reload

3. Open browser:
   http://127.0.0.1:8000/docs

# **Basic Code For The APIs**

In [135]:
# 1. Setup for Jupyter
nest_asyncio.apply()
app = FastAPI(title="FastAPI Assignment 5 - Day 6")

# 2. THE MISSING DATA (Inventory & Orders)
# Categories added specifically for Q5
products = [
    {"id": 1, "name": "Wireless Mouse", "price": 499, "category": "Electronics", "in_stock": True},
    {"id": 2, "name": "Notebook", "price": 99, "category": "Stationery", "in_stock": True},
    {"id": 3, "name": "USB Hub", "price": 799, "category": "Electronics", "in_stock": False},
    {"id": 4, "name": "Pen Set", "price": 49, "category": "Stationery", "in_stock": True},
    {"id": 5, "name": "Gaming Keyboard", "price": 1299, "category": "Electronics", "in_stock": True}
]

# Order list starts empty for your testing
orders = []

class OrderCreate(BaseModel):
    customer_name: str
    product_id: int


# **Question 1: Test the Search Endpoint ?**
- The search functionality in an API typically uses a keyword query parameter. To make it user-friendly, the search is implemented as case-insensitive and uses partial matching.

In [137]:
# Root
@app.get("/")
def root():
    return {"message": "FastAPI Day 6 — Search, Sort & Pagination API is running!"}


@app.get("/products/search")
def search_products(keyword: str):
    # Case-insensitive partial match logic
    results = [p for p in products if keyword.lower() in p['name'].lower()]
    
    if not results:
        return {"message": f"No products found for: {keyword}"}
    
    return {"keyword": keyword, "total_found": len(results), "products": results}

# **Question 2: Test All Sort Combinations**
- Sorting allows users to organize data based on their preference, such as price or alphabetical order.
- The API uses a sort_by parameter to define the field (Price or Name) and an order parameter (asc for ascending or desc for descending).
- If an invalid field like "category" is sent, the API returns an error to ensure data integrity.

In [139]:
@app.get("/products/sort")
def sort_products(sort_by: str = "price", order: str = "asc"):
    if sort_by not in ["price", "name"]:
        return {"error": "sort_by must be 'price' or 'name'"}
    
    reverse_order = (order == "desc")
    sorted_list = sorted(products, key=lambda p: p[sort_by], reverse=reverse_order)
    
    return {"sort_by": sort_by, "order": order, "products": sorted_list}

# **Question 3: Pagination Logic ?** 
- Pagination prevents the API from sending too much data at once. It uses a page number and a limit (items per page).
- We calculate the start index using the formula: $(page - 1) \times limit$. To find the total_pages, we use "ceiling division" so that any leftover items get their own page.

In [141]:
@app.get("/products/page")
def paginate_products(page: int = 1, limit: int = 2):
    start = (page - 1) * limit
    paged_products = products[start : start + limit]
    total_pages = -(-len(products) // limit) # Ceiling division
    
    return {"page": page, "limit": limit, "total_pages": total_pages, "products": paged_products}

# **Question 4: Search the Orders List ?**
- Similar to product search, we can search the orders list.
- This endpoint filters orders based on the customer_name.
- It returns a friendly message if no matching orders are found, rather than an empty list.

In [143]:
@app.get("/orders/search")
def search_orders(customer_name: str):
    results = [o for o in orders if customer_name.lower() in o['customer_name'].lower()]
    if not results:
        return {"message": f"No orders found for: {customer_name}"}
    return {"customer_name": customer_name, "total_found": len(results), "orders": results}

# **Question 5: Sort by Category Then Price ?**
- Advanced sorting can involve multiple levels.
- Here, we group products by category first (Stationery vs. Electronics).
- Within those categories, we sort them by price (cheapest first). Python’s sorted() function handles this easily by accepting a tuple as the sorting key.

In [145]:
@app.get("/products/sort-by-category")
def sort_by_category():
    # Sorts by category (A-Z) then price (Low-High)
    result = sorted(products, key=lambda p: (p['category'], p['price']))
    return {"products": result, "total": len(result)}

# **Question 6: Search + Sort + Paginate (The "Smart" Endpoint) ?**
- This is a high-performance "Browse" endpoint. It chains all three operations in a specific order:
- Filter (Keyword search)
- Sort (Price/Name)
- Paginate (Page/Limit).
- This allows the user to perform complex queries in a single API call.

In [147]:
@app.get("/products/browse")
def browse_products(
    keyword: str = None, 
    sort_by: str = "price", 
    order: str = "asc", 
    page: int = 1, 
    limit: int = 4
):
    result = products
    if keyword:
        result = [p for p in result if keyword.lower() in p['name'].lower()]
    
    if sort_by in ["price", "name"]:
        result = sorted(result, key=lambda p: p[sort_by], reverse=(order=="desc"))
        
    total = len(result)
    start = (page - 1) * limit
    paged = result[start : start + limit]
    
    return {
        "page": page, "total_found": total, 
        "total_pages": -(-total // limit), "products": paged
    }

# **Bonus: Paginate the Orders List ?**

In [149]:
# Bonus Task: Paginate the Orders List
@app.get("/orders/page")
def paginate_orders(page: int = 1, limit: int = 3):
    start = (page - 1) * limit
    paged_orders = orders[start : start + limit]
    total_pages = -(-len(orders) // limit) # Ceiling division logic
    
    return {
        "page": page, 
        "limit": limit, 
        "total_orders": len(orders),
        "total_pages": total_pages, 
        "orders": paged_orders
    }

# **Server Running Code**

In [176]:
import threading
import uvicorn
import nest_asyncio

nest_asyncio.apply()
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

# Start server in a separate thread
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

INFO:     Started server process [27160]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8004): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
